# 04 · Пары: DPO, ORPO, SimPO, KTO

Пары показывают модели и правильный ответ, и неправильный, отличающийся одним нарушением.
Общая основа — модель Брэдли–Терри: $P(y_w \succ y_l \mid x) = \sigma(r(x, y_w) - r(x, y_l))$.
Методы различаются тем, как награда $r$ выражается через саму языковую модель.

**DPO.** $r_\theta = \beta \log \frac{\pi_\theta(y \mid x)}{\pi_{\text{ref}}(y \mid x)}$, отсюда

$$
\mathcal{L}_{\text{DPO}} = -\log \sigma\!\Big(\beta \log \tfrac{\pi_\theta(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \beta \log \tfrac{\pi_\theta(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)}\Big).
$$

С LoRA референс бесплатен: та же модель с выключенным адаптером.

**ORPO.** Без референса: $\mathcal{L}_{\text{SFT}}(y_w) - \lambda \log \sigma\big(\log \tfrac{\text{odds}_\theta(y_w)}{\text{odds}_\theta(y_l)}\big)$,
$\text{odds}(y) = \tfrac{\pi(y)}{1 - \pi(y)}$.

**SimPO.** Без референса, награда — средний логарифм на токен, порог $\gamma$ требует запаса:
$-\log \sigma\big(\tfrac{\beta}{|y_w|} \log \pi_\theta(y_w) - \tfrac{\beta}{|y_l|} \log \pi_\theta(y_l) - \gamma\big)$.

**KTO.** Пары не нужны, каждый ответ с меткой. $v = \lambda_D \sigma(\beta (r_\theta - z_0))$ для
желательного и $\lambda_U \sigma(\beta (z_0 - r_\theta))$ для нежелательного, $z_0$ — средняя KL
до референса по батчу; $\mathcal{L} = \mathbb{E}[\lambda_y - v]$.

Стартуем от SFT-модели, а не от базы: это стандартный рецепт, SFT даёт манеру, пары двигают границу.
Адаптер SFT вливается в веса, и он же становится референсом.

In [ ]:
import sys
sys.path.insert(0, "../..")

from src import data, infer, metrics, report

from peft import LoraConfig, PeftModel
from trl import DPOConfig, DPOTrainer, KTOConfig, KTOTrainer
from trl.experimental.cpo import CPOConfig, CPOTrainer      # SimPO is CPO with loss_type="simpo"
from trl.experimental.orpo import ORPOConfig, ORPOTrainer

model, tokenizer = infer.load_model()
model = PeftModel.from_pretrained(model, report.RUNS / "sft-adapter").merge_and_unload()

train = data.load("train")
pairs = train.select_columns(["prompt", "chosen", "rejected"])
unpaired = data.to_kto(train)
print(pairs)
print(f"KTO: {len(unpaired)} примеров, хороших {sum(unpaired['label'])}")

Общие настройки одни: два прохода, эффективный батч 8, тот же адаптер, что в SFT. Отличия по делу:
KTO оценивает $z_0$ по соседям в батче и отвергает батч из одного примера, поэтому идёт по два
с накоплением четыре. SimPO без референса и без SFT-члена при шаге 5e-5 разваливается в бессвязный
текст, поэтому у него шаг 1e-5 и небольшой NLL-якорь `cpo_alpha`.

In [ ]:
lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    # attention and MLP projections of the language stack; the vision tower is excluded, there are no images
    target_modules=r"^(?!.*(visual|vision)).*(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$",
    use_rslora=True,
    task_type="CAUSAL_LM",
)

COMMON = dict(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    learning_rate=5e-5,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_length=2048,
    logging_steps=10,
    save_strategy="no",
    report_to=[],
    seed=42,
)
METHODS = {
    "dpo":   (DPOConfig,  DPOTrainer,  {"beta": 0.1}),
    "orpo":  (ORPOConfig, ORPOTrainer, {"beta": 0.1}),
    "simpo": (CPOConfig,  CPOTrainer,  {"loss_type": "simpo", "cpo_alpha": 0.05, "simpo_gamma": 0.5, "learning_rate": 1e-5}),
    "kto":   (KTOConfig,  KTOTrainer,  {"beta": 0.1, "per_device_train_batch_size": 2, "gradient_accumulation_steps": 4}),
}

In [ ]:
for name, (Config, Trainer, specific) in METHODS.items():
    print("═" * 78, name.upper())
    config = Config(output_dir=str(report.RUNS / name), **{**COMMON, **specific})
    trainer = Trainer(model=model, args=config, train_dataset=unpaired if name == "kto" else pairs,
                      processing_class=tokenizer, peft_config=lora)
    history = trainer.train()
    tuned = trainer.model
    print(f"loss {history.training_loss:.3f} | {infer.free(trainer)}")
    del trainer

    report.evaluate(tuned, tokenizer, name, note=f"{name} on top of SFT, 2 epochs")
    tuned.save_pretrained(report.RUNS / f"{name}-adapter")
    model = tuned.unload()
    del tuned
    print(infer.free())

report.show()

На что смотреть: методы на парах должны двигать судью и pref_acc; если при этом падает
`refusal_judge`, метод выучил из пар не то различие — в парах отказ стоит и в `chosen`, и
в `rejected`, и модель может принять сам факт отказа за признак плохого ответа.